In [71]:
from __future__ import annotations

import csv
import json
import shlex
import subprocess
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable, Union

import click
import pandas as pd
from Bio import SeqIO
from loguru import logger
import os
import pysam

In [3]:
raw = pd.read_csv("/public/home/bh2022044/sunda/Single-wastewater/Mock/05_ExtractBcReads/WithOutReference/mock_2.5k/barcodes.csv",  engine="python", header=None, dtype=str)

In [8]:
raw.iloc[1,]

0    AAACCCACGATGACTGT
Name: 1, dtype: object

In [11]:
dat = pd.read_csv("/public/home/bh2022044/sunda/Single-wastewater/Mock/05_ExtractBcReads/WithOutReference/mock_2.5k/barcodes.tsv", sep="\t", header=None)

In [28]:
dat.iloc[:, 0] = "barcode" + dat.iloc[:, 0].astype(str)

/tmp/ipykernel_34447/328335098.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0          barcode1
1          barcode2
2          barcode3
3          barcode4
4          barcode5
           ...     
3145    barcode3146
3146    barcode3147
3147    barcode3148
3148    barcode3149
3149    barcode3150
Name: 0, Length: 3150, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  dat.iloc[:, 0] = "barcode" + dat.iloc[:, 0].astype(str)


In [33]:
dat.to_csv("/public/home/bh2022044/sunda/Single-wastewater/dev_test/barcodes_reads.tsv", sep="\t", header=None, index=False)

In [37]:
raw = pd.read_csv("/public/home/bh2022044/sunda/Single-wastewater/dev_test/barcodes_reads.tsv", sep="\t", header=None, dtype=str)
raw = raw.dropna(how="all")
outdir = Path("/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/")
out_path = outdir / "barcodes.normalized.tsv"


In [38]:
raw.shape[1]

2

In [41]:
if raw.shape[1] >= 2:
    raw = raw.iloc[:, :2].copy()
    raw.columns = ["ReadPrefix", "Barcode"]
    header_mask = (
        raw["ReadPrefix"].str.lower().isin({"index", "idx", "read", "reads", "readprefix", "file"}) |
        raw["Barcode"].str.lower().isin({"barcode", "barcodes", "cellid", "cell"})
    )
    raw = raw.loc[~header_mask].copy()
    raw[["ReadPrefix", "Barcode"]].to_csv(out_path, sep="\t", index=False, header=False)
    prefix_map = dict(zip(raw["Barcode"].astype(str), raw["ReadPrefix"].astype(str)))
else:
    raise ValueError("barcodes file must be two tab-separated columns: read_prefix<TAB>barcode")

logger.info("Normalized barcode list: {}", out_path)

2026-07-08 15:27:58.774 | INFO     | __main__:<module>:14 - Normalized barcode list: /public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/barcodes.normalized.tsv


In [42]:
df = pd.read_csv("/public/home/bh2022044/sunda/Single-wastewater/Mock/05_ExtractBcReads/WithOutReference/mock_2.5k/counts.csv", sep=None, engine="python")
rename = {
    "cell": "CellID",
    "barcode": "CellID",
    "Barcode": "CellID",
    "umi": "Uminum",
    "UMI": "Uminum",
    "UmiNum": "Uminum",
    "reads": "ReadNum",
    "Reads": "ReadNum",
    "readnum": "ReadNum",
}

In [43]:
df

,CellID,Uminum,ReadNum
0,Barcode,1,1
1,AAACCCAATGAAACACT,3,6
2,AAACCCAATGAAACCAT,3,18
3,AAACCCAATGAAACCCA,1,4
4,AAACCCAATGAAACCCG,2,4
...,...,...,...
977382,TTTGTTGTACTTGGAAT,2,6
977383,TTTGTTGTACTTGGACG,7,16
977384,TTTGTTGTACTTTACAC,1,2
977385,TTTGTTGTACTTTAGGA,2,4


In [44]:
def normalize_barcode_list(barcodes_file: Path, outdir: Path) -> tuple[Path, dict[str, str]]:
    """
    Write two-column read-prefix/barcode TSV for Summary and return Barcode -> read prefix.

    Input is expected as two tab-separated columns:
      read_prefix<TAB>barcode
    """
    raw = pd.read_csv(barcodes_file, sep="\t", header=None, dtype=str)
    raw = raw.dropna(how="all")
    out_path = outdir / "barcodes.normalized.tsv"

    if raw.shape[1] >= 2:
        raw = raw.iloc[:, :2].copy()
        raw.columns = ["ReadPrefix", "Barcode"]
        header_mask = (
            raw["ReadPrefix"].str.lower().isin({"index", "idx", "read", "reads", "readprefix", "file"}) |
            raw["Barcode"].str.lower().isin({"barcode", "barcodes", "cellid", "cell"})
        )
        raw = raw.loc[~header_mask].copy()
        raw[["ReadPrefix", "Barcode"]].to_csv(out_path, sep="\t", index=False, header=False)
        prefix_map = dict(zip(raw["Barcode"].astype(str), raw["ReadPrefix"].astype(str)))
    else:
        raise ValueError("barcodes file must be two tab-separated columns: read_prefix<TAB>barcode")

    logger.info("Normalized barcode list: {}", out_path)
    return out_path, prefix_map

In [46]:
meta="/public/home/bh2022044/sunda/Single-wastewater/dev_test/barcodes_reads.tsv"
sample_dir = Path("/public/home/bh2022044/sunda/Single-wastewater/dev_test")
barcodes_txt, barcode_prefix_map = normalize_barcode_list(meta, sample_dir)


2026-07-08 16:51:13.978 | INFO     | __main__:normalize_barcode_list:25 - Normalized barcode list: /public/home/bh2022044/sunda/Single-wastewater/dev_test/barcodes.normalized.tsv


In [50]:
def summarize_contacts(
    json_dir: Path,
    purity_path: Path,
    outdir: Path,
    barcode_alias_map: dict[str, str] | None = None,
) -> tuple[pd.DataFrame, Path]:
    purity = pd.read_csv(purity_path, sep="\t")
    required = {"Barcode", "Purity", "ReadNum"}
    missing = required - set(purity.columns)
    if missing:
        raise ValueError(f"purity table missing columns {missing}; found {list(purity.columns)}")

    thresholded: dict[str, dict[str, int]] = defaultdict(dict)

    for json_file in sorted(json_dir.glob("*_p.json")):
        bin_name = json_file.name.removesuffix("_p.json")
        data, filter_bc, _drop_bc = filter_contact(load_json(json_file), purity, barcode_alias_map)
        filtered_total = sum(sum(barcodes.values()) for barcodes in filter_bc.values())
        filtered_by_contig = {contig: sum(barcodes.values()) for contig, barcodes in filter_bc.items()}
        threshold = filtered_total * 0.01

        for contig, value in filtered_by_contig.items():
            if value > threshold and value > 40:
                thresholded[bin_name][contig] = value
        logger.info(
            "[{}] filtered_total={} threshold={} kept_plasmids={}",
            bin_name, filtered_total, int(threshold), len(thresholded[bin_name])
        )

    rows = []
    for bin_name, plasmids in thresholded.items():
        for plasmid_id, readnum in plasmids.items():
            rows.append((bin_name, plasmid_id, readnum))

    df = pd.DataFrame(rows, columns=["Bin", "Plasmid", "ReadNum"])
    out_tsv = outdir / "plasmid_to_genome.tsv"
    #df.to_csv(out_tsv, sep="\t", index=False)
    logger.success("Wrote plasmid-to-genome links: {}", out_tsv)

    out_json = outdir / "plasmid_to_genome.json"
    with out_json.open("w", encoding="utf-8") as fh:
        json.dump({k: dict(v) for k, v in thresholded.items()}, fh, indent=4)
    return df, out_tsv

In [57]:
def filter_contact(
    data: dict[str, dict[str, int]],
    purity_df: pd.DataFrame,
    barcode_alias_map: dict[str, str] | None = None,
) -> tuple[dict[str, dict[str, int]], dict[str, dict[str, int]], dict[str, dict[str, list[float]]]]:
    drop_bc: dict[str, dict[str, list[float]]] = defaultdict(dict)
    filter_bc: dict[str, dict[str, int]] = defaultdict(dict)
    purity_dict = purity_df.set_index("Barcode")[["Purity", "ReadNum"]].to_dict(orient="index")
    barcode_alias_map = barcode_alias_map or {}

    for contig, barcodes_dict in data.items():
        for bc, count in barcodes_dict.items():
            info = purity_dict.get(bc)
            if not info and bc in barcode_alias_map:
                info = purity_dict.get(barcode_alias_map[bc])
            if not info:
                continue
            purity_value = float(info["Purity"])
            read_num = float(info["ReadNum"])
            exp_reads = max(5.0, read_num * purity_value * 0.005 / 100.0)

            if 80 <= purity_value <= 99:
                if count < exp_reads:
                    drop_bc[contig][bc] = [count, exp_reads, purity_value, read_num]
                else:
                    filter_bc[contig][bc] = count
            elif purity_value >= 100:
                filter_bc[contig][bc] = count
            else:
                drop_bc[contig][bc] = [count, exp_reads, purity_value, read_num]

    return data, filter_bc, drop_bc

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)

In [52]:
prefix_to_barcode = {prefix: barcode for barcode, prefix in barcode_prefix_map.items()}


In [58]:
link_df, link_tsv = summarize_contacts(
    Path("/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/BinReads2Plasimid"),
    Path("/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/purity.tsv"),
    sample_dir,
    barcode_alias_map=prefix_to_barcode,
)

2026-07-08 16:57:22.998 | INFO     | __main__:summarize_contacts:25 - [mock_bin5k_p_Bacillus] filtered_total=14918 threshold=149 kept_plasmids=1
2026-07-08 16:57:23.007 | INFO     | __main__:summarize_contacts:25 - [mock_bin5k_p_Ecoli] filtered_total=13822 threshold=138 kept_plasmids=2
2026-07-08 16:57:23.016 | INFO     | __main__:summarize_contacts:25 - [mock_bin5k_p_Kp] filtered_total=167447 threshold=1674 kept_plasmids=1
2026-07-08 16:57:23.023 | INFO     | __main__:summarize_contacts:25 - [mock_bin5k_p_Saures] filtered_total=47 threshold=0 kept_plasmids=0
2026-07-08 16:57:23.023 | SUCCESS  | __main__:summarize_contacts:38 - Wrote plasmid-to-genome links: /public/home/bh2022044/sunda/Single-wastewater/dev_test/plasmid_to_genome.tsv


In [59]:
list1 = ["Kp", "Bacillus", "Saures", "Ecoli"]

In [87]:
def NoName(path, purity):
    data1 = load_json(path)
    #print(data1.keys())
    
    drop_bc = defaultdict(lambda: defaultdict())
    filter_bc = defaultdict(lambda: defaultdict(int))
    
    # 预先构建 `Barcode` 到 `Purity` 和 `ReadNum` 的映射字典，避免循环内查询 DataFrame
    purity_dict = purity.set_index('Barcode')[['Purity', 'ReadNum']].to_dict(orient='index')
    
    # 遍历 `data1`
    for contig, barcodes_dict in data1.items():  # 用 `list()` 以支持 `pop()` 操作
        #print(contig)
        for bc in barcodes_dict.keys():  # 用 `list()` 以支持 `pop()` 操作
            if bc in purity_dict:  # 确保 `bc` 在 `purity_dict` 中
                purity_value = purity_dict[bc]['Purity']
                #print(purity_value)
                read_num = float(purity_dict[bc]['ReadNum'])
                exp_reads = max(5,float(read_num * purity_value * 0.005/100))
                #print(exp_reads)
                #print(f"{bc} purtiy is {purity_value} and read num is {read_num}")
                if purity_value <= 99 and purity_value >= 80:

                    #print(f"{bc} exp_read is {exp_reads}")
                    if barcodes_dict[bc] < exp_reads:
    
                        drop_bc[contig][bc] = [barcodes_dict[bc], exp_reads, purity_value, read_num]
                    else:
                        #if bc=="TCACATTGCTATAGAGT":
                           # print(data1[contig][bc], purity)
                        filter_bc[contig][bc] = data1[contig][bc]
                           
                elif purity_value >= 100:
                    #if bc=="TCACATTGCTATAGAGT":
                    #       print(data1[contig][bc], purity_value)
                    #print(path, contig)
                    filter_bc[contig][bc] = data1[contig][bc]
                else:
                    drop_bc[contig][bc] = [barcodes_dict[bc], exp_reads, purity_value, read_num]
    return data1, filter_bc, drop_bc

def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)

In [66]:
purity = pd.read_csv("/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/purity.tsv", sep="\t")

In [67]:
raw_list = list()
filter_list = list()
#thres_d = list()

raw_d = defaultdict(lambda: defaultdict())
filter_d = defaultdict(lambda: defaultdict())
thres_d = defaultdict(lambda: defaultdict())
for i in list1:
#for i in "Saure":
    #print(str(i))
    file = "/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/" + i.strip() + "_c_p.json"
    #file = "BinReads2Plasimid/mock_bin5k_p_Saures_p.json"
    print(file)
    if os.path.exists(file):
        print(file)
        data, filter_bc, drop_bc = NoName(file, purity)
        
        a = sum(sum(bc_dict.values()) for bc_dict in filter_bc.values())
    
        b = {contig: sum(barcodes.values()) for contig, barcodes in filter_bc.items()}
    
        raw_d[i.strip()] = a
        filter_d[i.strip()] = b
        
        threshold = a * 0.01
        print(i, threshold)
        for contig, value in b.items():
            if value > threshold and value > 40:
                thres_d[i.strip()][contig] = b[contig]
                #print(f"{file} {contig} {value} {threshold}")
        raw_list.append(a)
        filter_list.append(b)

/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Kp_c_p.json
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Kp_c_p.json
Kp 5077.6
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Bacillus_c_p.json
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Bacillus_c_p.json
Bacillus 493.91
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Saures_c_p.json
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Saures_c_p.json
Saures 14.76
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Ecoli_c_p.json
/public/home/bh2022044/sunda/Single-wastewater/dev_test/contact/mock_2.5k_valid/all_plasmid/contact_json/Ecoli_c_p.json

In [68]:
thres_d 

defaultdict(<function __main__.<lambda>()>,
            {'Kp': defaultdict(None, {'opera_contig_34': 507083}),
             'Bacillus': defaultdict(None, {'opera_contig_41': 49271}),
             'Saures': defaultdict(None,
                         {'opera_contig_34': 1290, 'opera_contig_41': 152}),
             'Ecoli': defaultdict(None,
                         {'opera_contig_34': 12499,
                          'opera_contig_51': 53961})})

In [78]:
list1 = ["Saures",]

for i in list1:
    bin2bc = defaultdict(lambda: defaultdict(int))
    file =  "/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/" + "mock_p_" + i + ".bam"
    print(file)
    if os.stat(file).st_size > 600:
        bam = pysam.AlignmentFile(file, "rb")
        for align in bam:
            bc = align.qname.split("_",1)[0]
            contig = align.reference_name
            bin2bc[contig][bc] +=1
        print(f"{file} is processed")
        outfile = "/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/" + "mock_bin5k_p_" + i + "_p.json"
        with open(outfile, "w") as json_file:
            json.dump(bin2bc, json_file, indent=4)
    else:
        logger.info(f"{file} is empty")

/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/mock_p_Saures.bam
/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/mock_p_Saures.bam is processed


[E::idx_find_and_load] Could not retrieve index file for '/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/mock_p_Saures.bam'


In [88]:
purity = pd.read_csv("~/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/purity.tsv", sep="\t")

In [89]:
raw_list = list()
filter_list = list()
#thres_d = list()

raw_d = defaultdict(lambda: defaultdict())
filter_d = defaultdict(lambda: defaultdict())
thres_d = defaultdict(lambda: defaultdict())
for i in list1:
#for i in "Saure":
    #print(str(i))
    file = "/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/" + "mock_bin5k_p_" + i + "_p.json"
    #file = "BinReads2Plasimid/mock_bin5k_p_Saures_p.json"
    print(file)
    if os.path.exists(file):
        print(file)
        data, filter_bc, drop_bc = NoName(file, purity)
        
        a = sum(sum(bc_dict.values()) for bc_dict in filter_bc.values())
    
        b = {contig: sum(barcodes.values()) for contig, barcodes in filter_bc.items()}
    
        raw_d[i.strip()] = a
        filter_d[i.strip()] = b
        
        threshold = a * 0.01
        print(i, threshold)
        for contig, value in b.items():
            if value > threshold and value > 40:
                thres_d[i.strip()][contig] = b[contig]
                #print(f"{file} {contig} {value} {threshold}")
        raw_list.append(a)
        filter_list.append(b)

/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/mock_bin5k_p_Saures_p.json
/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/check/mock_bin5k_p_Saures_p.json
Saures 15.860000000000001


In [90]:
filter_d

defaultdict(<function __main__.<lambda>()>,
            {'Saures': {'opera_contig_34': 1290,
              'opera_contig_41': 262,
              'opera_contig_51': 34}})

In [91]:
thres_d

defaultdict(<function __main__.<lambda>()>,
            {'Saures': defaultdict(None,
                         {'opera_contig_34': 1290, 'opera_contig_41': 262})})

In [93]:
raw_list = list()
filter_list = list()
#thres_d = list()

raw_d = defaultdict(lambda: defaultdict())
filter_d = defaultdict(lambda: defaultdict())
thres_d = defaultdict(lambda: defaultdict())
for i in list1:
#for i in "Saure":
    #print(str(i))
    file = "/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/BinReads2Plasimid/mock_bin5k_p_" + i.strip() + "_p.json"
    #file = "BinReads2Plasimid/mock_bin5k_p_Saures_p.json"
    print(file)
    if os.path.exists(file):
        print(file)
        data, filter_bc, drop_bc = NoName(file, purity)
        
        a = sum(sum(bc_dict.values()) for bc_dict in filter_bc.values())
    
        b = {contig: sum(barcodes.values()) for contig, barcodes in filter_bc.items()}
    
        raw_d[i.strip()] = a
        filter_d[i.strip()] = b
        
        threshold = a * 0.01
        print(i, threshold)
        for contig, value in b.items():
            if value > threshold and value > 40:
                thres_d[i.strip()][contig] = b[contig]
                #print(f"{file} {contig} {value} {threshold}")
        raw_list.append(a)
        filter_list.append(b)

/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/BinReads2Plasimid/mock_bin5k_p_Saures_p.json
/public/home/bh2022044/sunda/Single-wastewater/Mock/20260220_check_5k/06_ContactMap/BinReads2Plasimid/mock_bin5k_p_Saures_p.json
Saures 0.47000000000000003


In [94]:
thres_d

defaultdict(<function __main__.<lambda>()>, {})

In [95]:
filter_d

defaultdict(<function __main__.<lambda>()>,
            {'Saures': {'opera_contig_34': 36,
              'opera_contig_41': 10,
              'opera_contig_51': 1}})